# Train MFFT-Base on Kaggle
**Multi-Frequency Fusion Transformer — Base Variant (1.62M params)**

## Setup
1. **Settings → Accelerator**: GPU T4 x2 (or P100)
2. **Settings → Internet**: On
3. **Add Input** (attach all 11 datasets)
4. **Secrets → Add Secret**: `HF_TOKEN`

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
print(f"Cloned repo to {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn", "python-dotenv", "tqdm"], check=False)
print("Deps installed.")

In [ ]:
# Cell 2: Verify GPU & HF token
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "No GPU")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: HF_TOKEN not found: {e}")

In [ ]:
# Cell 3: Run training (MFFT-Base)
%run /kaggle/working/mfft_repo/kaggle_train_resumable.py

In [ ]:
# Cell 4: Plot training curves
import json, os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Load results from HF
try:
    from huggingface_hub import hf_hub_download
    results_path = hf_hub_download(
        repo_id="MohsinElis/mfft-checkpoints",
        filename=f"runs/{run_id}/logs/metrics.csv",
        repo_type="model", token=hf_token,
    )
    import csv
    epochs, train_loss, val_loss, train_acc, val_acc, macro_f1, auc = [], [], [], [], [], [], []
    with open(results_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row["epoch"]))
            train_loss.append(float(row["train_loss"]))
            val_loss.append(float(row["val_loss"]))
            train_acc.append(float(row["train_accuracy"]))
            val_acc.append(float(row["val_accuracy"]))
            macro_f1.append(float(row["val_macro_f1"]))
            auc.append(float(row["val_auc"]))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"MFFT-Base Training — {run_id}", fontsize=14)

    # Loss
    axes[0, 0].plot(epochs, train_loss, "b-", label="Train")
    axes[0, 0].plot(epochs, val_loss, "r-", label="Val")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy
    axes[0, 1].plot(epochs, train_acc, "b-", label="Train")
    axes[0, 1].plot(epochs, val_acc, "r-", label="Val")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Macro-F1
    axes[1, 0].plot(epochs, macro_f1, "g-", label="Val Macro-F1")
    axes[1, 0].set_title("Macro-F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # AUC
    axes[1, 1].plot(epochs, auc, "m-", label="Val AUC")
    axes[1, 1].set_title("AUC-ROC")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()

    # Summary
    best_idx = macro_f1.index(max(macro_f1))
    print(f"\nBest epoch: {epochs[best_idx]}")
    print(f"  Train acc: {train_acc[best_idx]:.2f}%")
    print(f"  Val acc:   {val_acc[best_idx]:.2f}%")
    print(f"  Val F1:    {macro_f1[best_idx]:.4f}")
    print(f"  Val AUC:   {auc[best_idx]:.4f}")
    print(f"  Best model epoch: {state.best_model_epoch}")
    print(f"  Status: {state.status}")
except Exception as e:
    print(f"Could not load metrics: {e}")
    print("Check HF repo for results.")

## Results

- Checkpoints: `https://huggingface.co/MohsinElis/mfft-checkpoints/tree/main/runs/<run_id>`
- Manifest: `https://huggingface.co/MohsinElis/mfft-master-manifest/blob/main/manifest/split_manifest.json`